# Fine-tune Pre-trained Nicheformer Model for Cell Type Annotation

In [ ]:
import os
import torch
import random
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader
import anndata as ad
import pandas as pd
import mygene

from nicheformer.models._nicheformer import Nicheformer
from nicheformer.models._nicheformer_fine_tune import NicheformerFineTune
from nicheformer.data.dataset import NicheformerDataset

In [ ]:
# Set a seed for reproducibility

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

## Configuration

Set up the configuration parameters for fine-tuning.

In [ ]:
config = {
    'data_path': '/media/rokny/DATA2/Sally/data/cell_type_annotation/adata_with_split_SEAAD.h5ad',  # Path to your AnnData file
    'technology_mean_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/model_means/SEAAD_mean_script.npy',  # Path to technology mean file
    'checkpoint_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/nicheformer.ckpt',  # Path to pre-trained model
    'output_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/benchmarking/cell_type_annotation/predictions_SEAAD.h5ad',  # Where to save results
    'output_dir': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/fine_tuned_models',  # Directory for checkpoints
    
    # Training parameters
    'batch_size': 9, 
    'max_seq_len': 1500,
    'aux_tokens': 30,
    'chunk_size': 1000,
    'num_workers': 4,
    'precision': 32,
    'max_epochs': 1,
    'lr': 1e-4,
    'warmup': 1,
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 10,
    
    # Model parameters
    'supervised_task': 'niche_classification',  # or whichever task
    'extract_layers': [11],  # Which layers to extract features from
    'function_layers': 'mean',  # Architecture of prediction head
    'dim_prediction': 1, # dim of the output vector
    'n_classes': 24,  # only for classification tasks
    'freeze': False,  # Whether to freeze backbone
    'reinit_layers': False,
    'extractor': False,
    'regress_distribution': False,
    'pool': 'mean',
    'predict_density': False,
    'ignore_zeros': False,
    'organ': 'unknown',
    'label': 'celltype_codes'  # The target variable to predict
}

## Load Data and Create Datasets

In [ ]:
model = ad.read_h5ad('/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/model_means/model.h5ad') 

print(model.var_names)
# Set random seed for reproducibility
pl.seed_everything(42)

# Load data
adata = ad.read_h5ad(config['data_path'])

technology_mean = np.load(config['technology_mean_path'])

In [ ]:
# Convert gene symbols to ensembl IDs

mg = mygene.MyGeneInfo()

gene_symbols = adata.var_names.tolist()

out = mg.querymany(
    gene_symbols,
    scopes="symbol",  
    fields="ensembl.gene",
    species="human"     
)

df = pd.DataFrame(out)

df["ensembl_id"] = df["ensembl"].apply(
    lambda x: x[0]["gene"] if isinstance(x, list) else (x["gene"] if isinstance(x, dict) else None)
)

mapping = df.set_index("query")["ensembl_id"].dropna().to_dict()

adata.var["gene_symbol"] = adata.var_names
adata.var_names = [mapping.get(g, g) for g in adata.var_names]
adata.var_names_make_unique()

# Subset adata to keep only the genes present in both model and adata
common_genes = model.var_names.intersection(adata.var_names)

# Subset adata to retain only the common genes
adata = adata[:, common_genes].copy()

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

print(adata)

In [ ]:
# Set metadata

adata.obs['modality'] = 4
adata.obs['specie'] = 5 
adata.obs['assay'] = 7

In [ ]:
# Split dataset into train and test 
from sklearn.model_selection import train_test_split

TARGET_COL = 'Subclass'

adata.obs['nicheformer_split'] = adata.obs['split']

split_counts = adata.obs['nicheformer_split'].value_counts()

print(f"Number of 'train' observations: {split_counts.get('train', 0)}")
print(f"Number of 'test' observations: {split_counts.get('test', 0)}")
print(f"Total number of observations: {adata.n_obs}")

for col in adata.obs.columns:
    adata.obs[col] = adata.obs[col].astype("category")

print(adata.obs[TARGET_COL])
adata.obs['celltype_codes'] = adata.obs[TARGET_COL].astype('category').cat.codes
print(adata.obs['celltype_codes'])

In [ ]:
import scipy

if scipy.sparse.issparse(adata.X):
    adata.X = adata.X.astype(np.float32)

# Create datasets
train_dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000),
    metadata_fields = {
        'obs': ['celltype_codes', 'modality', 'assay', 'specie'],
        # 'obsm': ['X_niche_0'],
    }
)

test_dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='test',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000),
    metadata_fields = {
        'obs': ['celltype_codes', 'modality', 'assay', 'specie'],
        # 'obsm': ['X_niche_0'],
    }
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

## Load Model and Set Up Fine-tuning

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)

# Create fine-tuning model
fine_tune_model = NicheformerFineTune(
    backbone=model,
    supervised_task=config['supervised_task'],
    extract_layers=config['extract_layers'],
    function_layers=config['function_layers'],
    lr=config['lr'],
    warmup=config['warmup'],
    max_epochs=config['max_epochs'],
    dim_prediction=config['dim_prediction'],
    n_classes=config['n_classes'],
    # baseline=config['baseline'],
    freeze=config['freeze'],
    reinit_layers=config['reinit_layers'],
    extractor=config['extractor'],
    regress_distribution=config['regress_distribution'],
    pool=config['pool'],
    predict_density=config['predict_density'],
    ignore_zeros=config['ignore_zeros'],
    organ=config.get('organ', 'unknown'),
    label=config['label'],
    without_context=True
)

# Configure trainer
trainer = pl.Trainer(
    max_epochs=config['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config['output_dir'],
    precision=config.get('precision', 32),
    gradient_clip_val=config.get('gradient_clip_val', 1.0),
    accumulate_grad_batches=config.get('accumulate_grad_batches', 10),
)

## Train and Evaluate Model

In [ ]:
# Train the model
print("Training the model...")
trainer.fit(
    model=fine_tune_model,
    train_dataloaders=train_loader,
)

# Test the model
print("Testing the model...")
test_results = trainer.test(
    model=fine_tune_model,
    dataloaders=test_loader
)

# Get predictions
print("Getting predictions...")
predictions = trainer.predict(fine_tune_model, dataloaders=test_loader)

In [ ]:
predictions = [torch.cat([p[0] for p in predictions]).cpu().numpy(),
               torch.cat([p[1] for p in predictions]).cpu().numpy(),
               torch.cat([p[2] for p in predictions]).cpu().numpy()]

In [ ]:
print(test_results)

In [ ]:
def metrics_calculation(pred_labels, true_labels, num_classes=None):
    if isinstance(pred_labels, np.ndarray):
        pred_labels = torch.tensor(pred_labels)
    if isinstance(true_labels, np.ndarray):
        true_labels = torch.tensor(true_labels)
    if num_classes is None:
        num_classes = int(torch.max(true_labels)) + 1

    pred_labels = pred_labels.view(-1).long()
    true_labels = true_labels.view(-1).long()

    pred_onehot = torch.nn.functional.one_hot(pred_labels, num_classes=num_classes)
    true_onehot = torch.nn.functional.one_hot(true_labels, num_classes=num_classes)

    tp = (pred_onehot & true_onehot).sum(dim=0).float()
    fp = (pred_onehot & (~true_onehot.bool())).sum(dim=0).float()
    fn = ((~pred_onehot.bool()) & true_onehot).sum(dim=0).float()

    tp_total = tp.sum()
    fp_total = fp.sum()
    fn_total = fn.sum()

    precision_micro = (tp_total / (tp_total + fp_total + 1e-8)).item()
    recall_micro = (tp_total / (tp_total + fn_total + 1e-8)).item()
    acc = (pred_labels == true_labels).float().mean().item()

    f1_per_class = (2 * tp / (2 * tp + fp + fn + 1e-8))
    f1_macro = f1_per_class.mean().item()

    return {
        'accuracy': acc,
        'f1': f1_macro,
        'precision': precision_micro,
        'recall': recall_micro
    }

In [ ]:
metrics = metrics_calculation(predictions[0], predictions[2], 24)

print(metrics)

In [ ]:
# Calculate PR-AUC and ROC-AUC metrics

import numpy as np
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.special import softmax

def macro_pr_roc_auc(y_true, proba):
    """
    y_true: (N,) integer class ids in [0, K-1]
    proba : (N, K) per-class probabilities (rows sum ~ 1)
    Returns (pr_auc_macro, roc_auc_macro)
    """
    y_true = np.asarray(y_true)
    proba  = np.asarray(proba)
    K = proba.shape[1]

    if y_true.min() < 0 or y_true.max() >= K:
        raise ValueError("y_true must be integer-coded in [0, K-1] matching proba columns.")
    if proba.ndim != 2 or proba.shape[0] != y_true.shape[0]:
        raise ValueError("proba must be (N, K) aligned with y_true (N,)")

    y_true_bin = label_binarize(y_true, classes=np.arange(K)) 

    pr_auc_macro  = average_precision_score(y_true_bin, proba, average="macro")
    roc_auc_macro = roc_auc_score(y_true, proba, average="macro", multi_class="ovr")
    return {"pr_auc_macro": pr_auc_macro, "roc_auc_macro": roc_auc_macro}

cats = adata.obs['Subclass'].astype('category').cat.categories
name_to_id = {name: i for i, name in enumerate(cats)}

labels_test = adata.obs.loc[adata.obs['nicheformer_split'] == 'test', 'Subclass']

y_true = labels_test.map(name_to_id).to_numpy(dtype=int)

proba = predictions[1] 

proba = np.asarray(proba)
proba = np.squeeze(proba)
if proba.ndim == 3 and proba.shape[-1] == 1:
    proba = proba[..., 0]

assert proba.ndim == 2, f"proba must be 2D (N,K); got {proba.shape}"
assert proba.shape[0] == y_true.shape[0], f"N mismatch: proba {proba.shape[0]} vs y_true {y_true.shape[0]}"

proba = softmax(proba, axis=1)

auc_metrics = macro_pr_roc_auc(y_true, proba)


In [ ]:
auc_metrics

## Save results for test set 

In [ ]:
adata_test = adata[adata.obs['nicheformer_split'] == 'test'].copy()

prediction_key = "predictions"

adata_test.obs["pred_labels"] = predictions[0]
adata_test.obs["true_labels"] = predictions[2]

for metric_name, value in test_results[0].items():
    adata_test.uns[f"{prediction_key}_metrics_{metric_name}"] = value

adata_test.write_h5ad(config['output_path'])

print(f"Results saved to {config['output_path']}")

## Load test set and predictions

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

adata = sc.read_h5ad("/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/benchmarking/cell_type_annotation/predictions_SEAAD.h5ad")

In [ ]:
adata.obs['pred_labels'] = adata.obs['pred_labels'].astype('category')
cats = adata.obs['Subclass'].astype('category').cat.categories
adata.obs['pred_labels_decoded'] = adata.obs['pred_labels'].map(lambda x: cats[x])

## Save results to npz file

In [ ]:
Model_name='nicheformer'
step='cell_type_annotation'
dataset='SEAAD'

In [ ]:
ACC, PRE, REC, F1 = float(metrics['accuracy']), float(metrics['precision']), float(metrics['recall']), float(metrics['f1'])
PR_AUC, ROC_AUC = float(auc_metrics['pr_auc_macro']), float(auc_metrics['roc_auc_macro'])

np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_{dataset}.npz",
    predictions=adata.obs['pred_labels_decoded'],
    labels=adata.obs['Subclass'],
    ACC=ACC, PRE=PRE, REC=REC, F1=F1, 
    PR_AUC=PR_AUC, ROC_AUC=ROC_AUC       
)